In [0]:
# Clean and standardize sales transaction data
silver_sales = spark.table(
    "workspace.indian_ecommerce_sales_analytics.bronze_sales"
)

print("Sales Row Count:", silver_sales.count())

display(silver_sales.limit(20))

Sales Row Count: 250000


Order_ID,Customer_ID,Product_ID,Order_Date,Order_Time,Delivery_Date,Quantity,Unit_Price,Order_Value,Shipping_Cost,Coupon_Code,Coupon_Discount,Total_Amount,Payment_Mode,Order_Status,Rating,Review_Text,City,State,Customer_Age,Customer_Age_Group,_source_file,_ingestion_timestamp
ORD0000000001,CUST00014303,PROD001017,2026-06-07,08:20:00,2026-06-10,1,2622.46,2622.46,0.0,null,0.0,2622.46,COD,Delivered,null,null,Chennai,Tamil Nadu,43,36-45,sales.csv,2026-09-10T06:30:13.484Z
ORD0000000002,CUST00034253,PROD000703,2025-02-04,20:05:00,2025-02-07,1,90258.0,90258.0,0.0,null,0.0,90258.0,COD,Delivered,4.0,Nice quality,Durgapur,West Bengal,22,18-25,sales.csv,2026-09-10T06:30:13.484Z
ORD0000000003,CUST00000421,PROD000318,2026-03-12,14:59:00,2026-03-19,1,3536.5,3536.5,0.0,null,0.0,3536.5,UPI,Processing,null,null,Patiala,Punjab,41,36-45,sales.csv,2026-09-10T06:30:13.484Z
ORD0000000004,CUST00026507,PROD000213,2025-01-27,14:33:00,2025-02-02,1,34081.56,34081.56,0.0,FLAT50,50.0,34031.56,Credit Card,Delivered,null,null,Rajkot,Gujarat,29,26-35,sales.csv,2026-09-10T06:30:13.484Z
ORD0000000005,CUST00009418,PROD001437,2024-06-30,07:19:00,2024-07-02,2,69021.17,138042.34,0.0,null,0.0,138042.34,UPI,Delivered,null,null,Surat,Gujarat,39,36-45,sales.csv,2026-09-10T06:30:13.484Z
ORD0000000006,CUST00033802,PROD000121,2025-09-18,11:56:00,2025-09-23,1,4489.77,4489.77,0.0,FLAT50,50.0,4439.77,UPI,Returned,null,null,Agra,UP,50,46-55,sales.csv,2026-09-10T06:30:13.484Z
ORD0000000007,CUST00027469,PROD001308,2026-03-01,10:32:00,2026-03-05,2,33620.99,67241.98,0.0,null,0.0,67241.98,COD,Processing,null,null,Dwarka,Delhi,36,36-45,sales.csv,2026-09-10T06:30:13.484Z
ORD0000000008,CUST00017326,PROD001392,2024-06-19,17:46:00,2024-06-23,2,1161.62,2323.24,0.0,SAVE10,232.32,2090.92,Credit Card,Delivered,null,null,Delhi NCR,Delhi,20,18-25,sales.csv,2026-09-10T06:30:13.484Z
ORD0000000009,CUST00009627,PROD001876,2025-02-12,12:19:00,2025-02-15,1,22456.69,22456.69,0.0,SAVE10,2245.67,20211.02,Credit Card,Delivered,null,null,Agra,UP,29,26-35,sales.csv,2026-09-10T06:30:13.484Z
ORD0000000010,CUST00012719,PROD000214,2024-10-19,07:45:00,2024-10-25,1,86647.71,86647.71,0.0,null,0.0,86647.71,UPI,Delivered,5.0,Highly recommended!,Surat,Gujarat,39,36-45,sales.csv,2026-09-10T06:30:13.484Z


In [0]:
#Cleaning the text columns
from pyspark.sql import functions as F
string_columns = [
    "Order_ID",
    "Customer_ID",
    "Product_ID",
    "Coupon_Code",
    "Payment_Mode",
    "Order_Status",
    "Review_Text",
    "City",
    "State",
    "Customer_Age_Group"
]

for col_name in string_columns:
    silver_sales = silver_sales.withColumn(
        col_name,
        F.trim(F.col(col_name))
    )

In [0]:
silver_sales = silver_sales.withColumn(
    "Payment_Mode",
    F.initcap(F.lower(F.col("Payment_Mode")))
)
#Different types of payment methods and count
display(
    silver_sales
    .groupBy("Payment_Mode")
    .count()
    .orderBy(F.desc("count"))
)

Payment_Mode,count
Upi,128474
Cod,82093
Debit Card,28856
Credit Card,10577


In [0]:
silver_sales = silver_sales.withColumn(
    "Order_Status",
    F.initcap(F.lower(F.col("Order_Status")))
)
#Order status and count
display(
    silver_sales
    .groupBy("Order_Status")
    .count()
    .orderBy(F.desc("count"))
)

Order_Status,count
Delivered,200139
Cancelled,12507
Returned,12493
Shipped,12459
Processing,12402


In [0]:
#Handling coupon codes
silver_sales = silver_sales.withColumn(
    "Coupon_Code",
    F.when(
        F.col("Coupon_Code").isNull() |
        (F.col("Coupon_Code") == ""),
        "NO_COUPON"
    ).otherwise(
        F.upper(F.col("Coupon_Code"))
    )
)
display(
    silver_sales
    .groupBy("Coupon_Code")
    .count()
    .orderBy(F.desc("count"))
)

Coupon_Code,count
NO_COUPON,199815
SAVE10,24997
DIWALI100,12609
FLAT50,12579


In [0]:
#Converting numeric columns from diff datatypes
silver_sales = (
    silver_sales

    .withColumn("Quantity", F.col("Quantity").cast("int"))
    .withColumn("Unit_Price", F.col("Unit_Price").cast("double"))
    .withColumn("Order_Value", F.col("Order_Value").cast("double"))
    .withColumn("Shipping_Cost", F.col("Shipping_Cost").cast("double"))
    .withColumn("Coupon_Discount", F.col("Coupon_Discount").cast("double"))
    .withColumn("Total_Amount", F.col("Total_Amount").cast("double"))
    .withColumn("Customer_Age", F.col("Customer_Age").cast("int"))
    .withColumn("Rating", F.col("Rating").cast("double"))
)

In [0]:
#Converting dates
silver_sales = (
    silver_sales
    .withColumn(
        "Order_Date",
        F.to_date(F.col("Order_Date"))
    )
    .withColumn(
        "Delivery_Date",
        F.to_date(F.col("Delivery_Date"))
    )
)
#Converting order time
silver_sales = silver_sales.withColumn(
    "Order_Time",
    F.to_timestamp(F.col("Order_Time"), "HH:mm:ss")
)
silver_sales.select("Order_Time").show(10, False)

+-------------------+
|Order_Time         |
+-------------------+
|1970-01-01 08:20:00|
|1970-01-01 20:05:00|
|1970-01-01 14:59:00|
|1970-01-01 14:33:00|
|1970-01-01 07:19:00|
|1970-01-01 11:56:00|
|1970-01-01 10:32:00|
|1970-01-01 17:46:00|
|1970-01-01 12:19:00|
|1970-01-01 07:45:00|
+-------------------+
only showing top 10 rows


In [0]:
#Checking Invalid quantity count
invalid_quantity = silver_sales.filter(
    F.col("Quantity") <= 0
)

print(
    "Invalid Quantity Records:",
    invalid_quantity.count()
)
#checking invalid price records
invalid_unit_price = silver_sales.filter(
    F.col("Unit_Price") < 0
)

print(
    "Invalid Unit Price Records:",
    invalid_unit_price.count()
)

#Checking order value
invalid_order_value = silver_sales.filter(
    F.col("Order_Value") < 0
)

print(
    "Invalid Order Value Records:",
    invalid_order_value.count()
)
#and also coupon 
invalid_coupon_discount = silver_sales.filter(
    F.col("Coupon_Discount") < 0
)

print(
    "Invalid Coupon Discount Records:",
    invalid_coupon_discount.count()
)

# invalid ratings
invalid_rating = silver_sales.filter(
    (F.col("Rating") < 0) |
    (F.col("Rating") > 5)
)

print(
    "Invalid Rating Records:",
    invalid_rating.count()
)

#checking invalid delivery dates 
invalid_delivery_date = silver_sales.filter(
    F.col("Delivery_Date") < F.col("Order_Date")
)

print(
    "Invalid Delivery Date Records:",
    invalid_delivery_date.count()
)

Invalid Quantity Records: 0
Invalid Unit Price Records: 0
Invalid Order Value Records: 0
Invalid Coupon Discount Records: 0
Invalid Rating Records: 0
Invalid Delivery Date Records: 0


In [0]:
#Recalculating Transaction amounts
silver_sales = silver_sales.withColumn(
    "Calculated_Total_Amount",
    F.round(
        F.col("Order_Value")
        - F.coalesce(F.col("Coupon_Discount"), F.lit(0.0))
        + F.coalesce(F.col("Shipping_Cost"), F.lit(0.0)),
        2
    )
)

In [0]:
silver_sales = silver_sales.withColumn(
    "Amount_Difference",
    F.round(
        F.col("Total_Amount") -
        F.col("Calculated_Total_Amount"),
        2
    )
)
display(
    silver_sales
    .filter(F.abs(F.col("Amount_Difference")) > 0.01)
    .select(
        "Order_ID",
        "Order_Value",
        "Coupon_Discount",
        "Shipping_Cost",
        "Total_Amount",
        "Calculated_Total_Amount",
        "Amount_Difference"
    )
    .limit(20)
)

Order_ID,Order_Value,Coupon_Discount,Shipping_Cost,Total_Amount,Calculated_Total_Amount,Amount_Difference


In [0]:
silver_sales = silver_sales.withColumn(
    "dq_negative_total_amount",
    F.col("Total_Amount") < 0
)
print(
    "Negative Total Amount Records:",
    silver_sales
    .filter(F.col("dq_negative_total_amount"))
    .count()
)

Negative Total Amount Records: 5


In [0]:
#Amount validation Flag
silver_sales = silver_sales.withColumn(
    "dq_amount_mismatch",
    F.abs(F.col("Amount_Difference")) > 0.01
)

In [0]:
#Checking delivery dates
silver_sales = silver_sales.withColumn(
    "Delivery_Days",
    F.datediff(
        F.col("Delivery_Date"),
        F.col("Order_Date")
    )
)

display(
    silver_sales
    .select(
        "Order_ID",
        "Order_Date",
        "Delivery_Date",
        "Delivery_Days"
    )
    .limit(20)
)

Order_ID,Order_Date,Delivery_Date,Delivery_Days
ORD0000000001,2026-06-07,2026-06-10,3
ORD0000000002,2025-02-04,2025-02-07,3
ORD0000000003,2026-03-12,2026-03-19,7
ORD0000000004,2025-01-27,2025-02-02,6
ORD0000000005,2024-06-30,2024-07-02,2
ORD0000000006,2025-09-18,2025-09-23,5
ORD0000000007,2026-03-01,2026-03-05,4
ORD0000000008,2024-06-19,2024-06-23,4
ORD0000000009,2025-02-12,2025-02-15,3
ORD0000000010,2024-10-19,2024-10-25,6


In [0]:
#Date Dimentions
silver_sales = (
    silver_sales

    .withColumn(
        "Year",
        F.year("Order_Date")
    )

    .withColumn(
        "Month",
        F.month("Order_Date")
    )

    .withColumn(
        "Month_Name",
        F.date_format("Order_Date", "MMMM")
    )

    .withColumn(
        "Quarter",
        F.quarter("Order_Date")
    )

    .withColumn(
        "Year_Month",
        F.date_format("Order_Date", "yyyy-MM")
    )

    .withColumn(
        "Day",
        F.dayofmonth("Order_Date")
    )

    .withColumn(
        "Day_of_Week",
        F.date_format("Order_Date", "EEEE")
    )
)


In [0]:
#Coupon Flag
silver_sales = silver_sales.withColumn(
    "Has_Coupon",
    F.when(
        F.col("Coupon_Code") == "NO_COUPON",
        False
    ).otherwise(True)
)
# display(silver_sales).limit(10)

In [0]:
tier_1_cities = [
    "Mumbai",
    "Delhi",
    "Bengaluru",
    "Bangalore",
    "Hyderabad",
    "Chennai",
    "Kolkata",
    "Pune",
    "Ahmedabad"
]

tier_2_cities = [
    "Jaipur",
    "Lucknow",
    "Kanpur",
    "Nagpur",
    "Indore",
    "Bhopal",
    "Visakhapatnam",
    "Vijayawada",
    "Surat",
    "Coimbatore",
    "Kochi",
    "Chandigarh",
    "Patna",
    "Vadodara",
    "Nashik",
    "Agra",
    "Varanasi",
    "Ranchi",
    "Bhubaneswar",
    "Dehradun"
]

silver_sales = silver_sales.withColumn(
    "City_Tier",
    F.when(
        F.col("City").isin(tier_1_cities),
        "Tier 1"
    )
    .when(
        F.col("City").isin(tier_2_cities),
        "Tier 2"
    )
    .otherwise("Tier 3")
)

display(
    silver_sales
    .groupBy("City_Tier")
    .count()
    .orderBy("City_Tier")
)

City_Tier,count
Tier 1,37644
Tier 2,58272
Tier 3,154084


In [0]:
#Foreign-Key Validating
bronze_customers = spark.table(
    "workspace.indian_ecommerce_sales_analytics.silver_customers"
)

missing_customers = (
    silver_sales
    .select("Customer_ID")
    .distinct()
    .join(
        bronze_customers.select("Customer_ID").distinct(),
        on="Customer_ID",
        how="left_anti"
    )
)

print(
    "Sales records with missing Customer IDs:",
    missing_customers.count()
)


Sales records with missing Customer IDs: 0


In [0]:
#product FK Validating
bronze_products = spark.table(
    "workspace.indian_ecommerce_sales_analytics.silver_products"
)

missing_products = (
    silver_sales
    .select("Product_ID")
    .distinct()
    .join(
        bronze_products.select("Product_ID").distinct(),
        on="Product_ID",
        how="left_anti"
    )
)

print(
    "Sales records with missing Product IDs:",
    missing_products.count()
)

Sales records with missing Product IDs: 0


In [0]:
#Data quality flags
silver_sales = (
    silver_sales

    .withColumn(
        "dq_invalid_quantity",
        F.col("Quantity") <= 0
    )

    .withColumn(
        "dq_invalid_unit_price",
        F.col("Unit_Price") < 0
    )

    .withColumn(
        "dq_invalid_order_value",
        F.col("Order_Value") < 0
    )

    .withColumn(
        "dq_invalid_coupon_discount",
        F.col("Coupon_Discount") < 0
    )

    .withColumn(
        "dq_invalid_rating",
        (F.col("Rating") < 0) |
        (F.col("Rating") > 5)
    )

    .withColumn(
        "dq_invalid_delivery_date",
        F.col("Delivery_Date") < F.col("Order_Date")
    )
)

In [0]:
silver_sales = silver_sales.select(
    "Order_ID",
    "Customer_ID",
    "Product_ID",
    "Order_Date",
    "Order_Time",
    "Delivery_Date",
    "Delivery_Days",
    "Quantity",
    "Unit_Price",
    "Order_Value",
    "Shipping_Cost",
    "Coupon_Code",
    "Coupon_Discount",
    "Total_Amount",
    "Calculated_Total_Amount",
    "Amount_Difference",
    "Payment_Mode",
    "Order_Status",
    "Rating",
    "Review_Text",
    "City",
    "State",
    "Customer_Age",
    "Customer_Age_Group",
    "Year",
    "Month",
    "Month_Name",
    "Quarter",
    "Year_Month",
    "Day",
    "Day_of_Week",
    "Has_Coupon",
    "City_Tier",
    "dq_negative_total_amount",
    "dq_amount_mismatch",
    "dq_invalid_quantity",
    "dq_invalid_unit_price",
    "dq_invalid_order_value",
    "dq_invalid_coupon_discount",
    "dq_invalid_rating",
    "dq_invalid_delivery_date"
)
print(
    "Final Silver Sales Count:",
    silver_sales.count()
)

print(
    "Distinct Order IDs:",
    silver_sales
    .select("Order_ID")
    .distinct()
    .count()
)

Final Silver Sales Count: 250000
Distinct Order IDs: 250000


In [0]:
#Final checking null values
null_report = silver_sales.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in silver_sales.columns
])

display(null_report)

Order_ID,Customer_ID,Product_ID,Order_Date,Order_Time,Delivery_Date,Delivery_Days,Quantity,Unit_Price,Order_Value,Shipping_Cost,Coupon_Code,Coupon_Discount,Total_Amount,Calculated_Total_Amount,Amount_Difference,Payment_Mode,Order_Status,Rating,Review_Text,City,State,Customer_Age,Customer_Age_Group,Year,Month,Month_Name,Quarter,Year_Month,Day,Day_of_Week,Has_Coupon,City_Tier,dq_negative_total_amount,dq_amount_mismatch,dq_invalid_quantity,dq_invalid_unit_price,dq_invalid_order_value,dq_invalid_coupon_discount,dq_invalid_rating,dq_invalid_delivery_date
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,129970,129970,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,129970,0


In [0]:
#Saving solver table
(
    silver_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.indian_ecommerce_sales_analytics.silver_sales"
    )
)

In [0]:
%sql
SELECT COUNT(*) AS total_sales_records
FROM workspace.indian_ecommerce_sales_analytics.silver_sales;

total_sales_records
250000


In [0]:
%sql
SELECT
    Order_Status,
    COUNT(*) AS orders
FROM workspace.indian_ecommerce_sales_analytics.silver_sales
GROUP BY Order_Status
ORDER BY orders DESC;

Order_Status,orders
Delivered,200139
Cancelled,12507
Returned,12493
Shipped,12459
Processing,12402
